# Precipitation forecast vs. observation: pattern, location, scale, rotation, intensity

No objects or bounding boxes. A network looks at both fields together and estimates the **transform** that maps the observation (A) onto the forecast (B): shift, zoom, rotation, plus the rain-amount ratio. It then compares the fields **after undoing it**.

| score | meaning | perfect |
|---|---|---|
| **pattern** | correlation of (smoothed, log) rain once shift/size/rotation/amount errors are removed | 1 |
| **location** | `1 / (1 + d / 0.1)`, `d` = shift as a fraction of the domain | 1 |
| **scale** | smaller / larger size | 1 |
| **rotation** | `1 / (1 + angle / 20 deg)` | 1 |
| **intensity** | smaller / larger rain amount | 1 |
| **overall** | geometric mean | 1 |

Training is **self-supervised**: real rain fields (ERA5) are shifted / zoomed / rotated / rescaled by known amounts and distorted like a forecast would be (warping, smoothing, spurious and missing cells, part of the pattern replaced), so every label is exact. A brute-force classical registration and SAL-style moments are evaluated alongside as baselines.

**Runtime -> Change runtime type -> T4 GPU.**

In [ ]:
%pip install -q "zarr>=3" gcsfs

# ---- settings ----
GITHUB_USER = "your-github-id"   # <- your GitHub ID (or upload pattern-sim.zip to /content)
REPO = "pattern-sim"
USE_DRIVE = True                 # keep the downloaded fields + checkpoints in Google Drive
YEARS = range(2016, 2024)        # ERA5 years (May-Oct, 00/06/12/18 UTC); the last year is the test set

import os, sys, subprocess
REPO_DIR = f"/content/{REPO}"
if not os.path.isdir(REPO_DIR):
    if os.path.exists(f"/content/{REPO}.zip"):
        subprocess.run(["unzip", "-q", f"/content/{REPO}.zip", "-d", "/content"], check=True)
    else:
        subprocess.run(["git", "clone", f"https://github.com/{GITHUB_USER}/{REPO}.git", REPO_DIR], check=True)
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/pattern-sim"
else:
    WORK = "/content/work"
os.makedirs(WORK, exist_ok=True)

import torch, pandas as pd
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to GPU")

In [ ]:
# self-test on fake fields (no download), about a minute
!python -m pytest -q tests

## 1. Data: ERA5 hourly precipitation, East Asia / West Pacific (0-60N, 90-180E, 0.25 deg)
Read anonymously from Google's public ARCO-ERA5 store. Each hour is a full-globe chunk on the server, so the time depends on the number of hours (8 years x 184 days x 4 = ~5,900); a first run of a few minutes up to ~20 min is normal, then it is cached in Drive. To try things quickly, set `YEARS = range(2021, 2024)` first.

Below: synthetic training pairs. Top = observation crop (128 x 128 grid points = 32 deg), bottom = the "forecast" made from it, titled with its exact labels.

In [ ]:
from precipsim.data import download_era5, FieldPairSynth
from precipsim.viz import show_pairs

bank = download_era5(f"{WORK}/era5_ea_{YEARS.start}-{YEARS.stop - 1}_MJJASO.npz", years=YEARS)
DEV = "cuda"
train_bank = bank.select_years(YEARS.start, YEARS.stop - 2).to(DEV)
test_bank = bank.select_years(YEARS.stop - 1, YEARS.stop - 1).to(DEV)
print(tuple(bank.precip.shape), "|", len(train_bank), "training fields,", len(test_bank), "test fields")

show_pairs(FieldPairSynth(train_bank).sample(8), n=8);

## 2. Baselines (no training)
On 512 fixed test pairs: exhaustive zoom/rotation search + phase correlation, SAL-style moments (centroid shift, amount ratio), and plain gridpoint correlation as a pattern score.

In [ ]:
from precipsim.evaluate import evaluate, summary_table

test_synth = FieldPairSynth(test_bank)
summary_table(evaluate(None, test_synth, n_pairs=512, baseline=True))

## 3. Train, or reuse the saved model
The model is saved to `WORK/runs/precip/precipsim.pt` (in Google Drive) at every evaluation and at the end of training. When a saved model exists, this cell shows how long it was trained and its test errors, then asks:
* **u**: use it as is (no training)
* **c**: continue training from its weights for `steps` more steps (a lower `lr`, e.g. `1e-4`, usually works better here)
* **s**: train a new model from scratch

Before **c** or **s** overwrite the saved model, a dated copy (`precipsim_YYYYmmdd-HHMMSS.pt`) is kept next to it. To skip the question, pick `MODE` in the form. Training from scratch takes ~15-25 min on a T4.

In [ ]:
from precipsim.train import Config, checkpoint_path, prepare_model

MODE = "ask"  # @param ["ask", "use", "continue", "scratch"]
cfg = Config(out_dir=f"{WORK}/runs/precip", steps=8000)  # when continuing, e.g. steps=4000, lr=1e-4
CKPT = checkpoint_path(cfg)
model, history = prepare_model(cfg, train_bank, test_bank, mode=MODE)

In [ ]:
if history:
    h = pd.DataFrame(history).set_index("step")
    h[["shift_err_px", "scale_err_pct", "rot_err_deg", "amp_err_pct", "pattern_mae"]].plot(subplots=True, layout=(1, 5), figsize=(16, 3), marker="o");
summary_table(evaluate(model, test_synth, n_pairs=1024, baseline=True))

## 4. Scenarios: Typhoon Khanun south of Korea, 2023-08-09 12 UTC
Synthetic "forecasts" with known errors, then **real** differences: the observation 6 h / 24 h earlier used as a forecast (persistence), and an unrelated date. `corr_raw` is the plain gridpoint correlation for comparison (double penalty).

In [ ]:
from precipsim.infer import FieldScorer
from precipsim.viz import show_result

scorer = FieldScorer.from_checkpoint(CKPT)
synth = FieldPairSynth(test_bank)
WHEN, LAT, LON, KM = "2023-08-09T12", 33.0, 128.0, 25.0   # case, domain centre, ~grid spacing in km
i, c = test_bank.index_of(WHEN), test_bank.px(LAT, LON)

cases = {
    "identical": synth.make_pair(i, c),
    "shifted 12 pts (~300 km) east": synth.make_pair(i, c, shift_px=(12, 0)),
    "shifted ~8 pts south-west": synth.make_pair(i, c, shift_px=(-6, -6)),
    "rotated 20 deg ccw": synth.make_pair(i, c, rotation_ccw_deg=20),
    "1.3x larger": synth.make_pair(i, c, scale=1.3),
    "2x more rain": synth.make_pair(i, c, intensity=2.0),
    "all four at once": synth.make_pair(i, c, shift_px=(12, 0), rotation_ccw_deg=20, scale=1.3, intensity=2.0),
    "persistence: obs 6 h earlier": synth.make_pair(i, c, j=i - 1),
    "persistence: obs 24 h earlier": synth.make_pair(i, c, j=i - 4),
    "unrelated date (2023-06-15)": synth.make_pair(i, c, j=test_bank.index_of("2023-06-15T00")),
}
COLS = ["pattern", "location", "scale", "rotation", "intensity", "overall",
        "shift_km", "scale_ratio", "rotation_deg", "intensity_ratio", "corr_raw"]
rows = {}
for name, (obs, fcst) in cases.items():
    r = scorer.score(obs, fcst, grid_km=KM, return_maps=True)
    show_result(r, title=name)
    rows[name] = {k: round(r[k], 2) for k in COLS}
pd.DataFrame(rows).T

### Persistence over the whole test season
Real field pairs, no synthetic transform: the observation `lead` hours earlier as the forecast, on the same Korea-centred domain (only wet cases). Pattern and location should fall with lead time; `corr_raw` falls faster because it double-penalises displacement.

In [ ]:
import numpy as np

crop = lambda k: test_bank.crop(k, LAT, LON, 128)
rows = {}
for hours in (6, 12, 24, 48):
    s = hours // 6
    ks = [k for k in range(s, len(test_bank)) if test_bank.time[k] - test_bank.time[k - s] == np.timedelta64(hours, "h")]
    obs, fc = np.stack([crop(k) for k in ks]), np.stack([crop(k - s) for k in ks])
    wet = (obs > 0.2).mean((1, 2)) > 0.05
    r = scorer.score_batch(obs[wet], fc[wet], grid_km=KM)
    rows[f"{hours} h"] = {"cases": int(wet.sum()), **{k: float(np.median(r[k])) for k in COLS}}
pd.DataFrame(rows).T.round(2)

## 5. Your own data
Two 2D arrays on the same grid, row 0 = north (otherwise `north_up=False`), rain in mm per `accum_hours`. Any grid size works. Because training needs no labels, you can also **fine-tune on your own observations** (radar, GPM, analyses): the forecast-like pairs are synthesised from them.

In [ ]:
# import xarray as xr
# obs  = xr.open_dataset("obs.nc")["precip"].sel(time="2023-08-09T12").values
# fcst = xr.open_dataset("fcst.nc")["precip"].sel(time="2023-08-09T12").values
# r = scorer.score(obs, fcst, accum_hours=1, grid_km=12, return_maps=True)
# show_result(r); {k: v for k, v in r.items() if k != "maps"}
#
# Fine-tune / retrain on your own observations, (N, H, W) mm/h with H, W >= 128:
# from precipsim.data import FieldBank
# mine = FieldBank.from_array(obs_stack_mmh, time=obs_times)
# model, history = train(Config(out_dir=f"{WORK}/runs/mine"), mine.select_years(2019, 2022), mine.select_years(2023, 2023))

## 6. Save
Next time you run this notebook, step 3 finds the saved model and asks whether to use it, continue training it, or start from scratch.

In [ ]:
# The model is already saved in CKPT. Keep a dated copy that later runs never overwrite.
# With USE_DRIVE = False, also download it: /content is wiped when the Colab runtime ends.
from precipsim.train import describe_checkpoint, snapshot_checkpoint

print(describe_checkpoint(CKPT))
print("dated copy:", snapshot_checkpoint(CKPT))
if not USE_DRIVE:
    from google.colab import files
    files.download(CKPT)

## Knobs
* **Score shapes** (no retraining): `FieldScorer.from_checkpoint(path, d0=0.1, rot0_deg=20, refine=1)`. `d0` = shift (fraction of the domain) where location = 0.5, `rot0_deg` = angle where rotation = 0.5, `refine` = extra alignment passes.
* **How hard the synthetic forecasts are** (retrain): `Config(distortion=1.0)`; ranges of shift / zoom / rotation / amount are the `FieldPairSynth` arguments in `precipsim/data.py`.
* **Uncertainty**: `unc_shift`, `unc_scale_pct`, `unc_rotation_deg` are the network's own expected errors, large e.g. for rotation of a round typhoon or for scattered convection.